# Classical Image Analysis and Registration

This notebook demonstrates image registration, template matching, automatic thresholding, line detection, and seeded segmentation with OpenCV.



In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    raise FileNotFoundError("Run the notebook from the repository root; data/ was not found.")


## 1. Landmark-based affine image registration

Register a moving MRI image to a fixed MRI image by selecting three corresponding landmark pairs and estimating an affine transformation.


In [ ]:
# read target  and source
target_bgr = cv2.imread(str(DATA_DIR / "MRI.jpg"))    
source_bgr = cv2.imread(str(DATA_DIR / "MRI2.jpg"))   
# convert to RGB for plotting
target = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2RGB)
source = cv2.cvtColor(source_bgr, cv2.COLOR_BGR2RGB)
# grayscale
target_g = cv2.cvtColor(target, cv2.COLOR_RGB2GRAY)
source_g = cv2.cvtColor(source, cv2.COLOR_RGB2GRAY)
# display
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(target, cmap="gray"); plt.title("Target (MRI.jpg)"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(source, cmap="gray"); plt.title("Source (MRI2.jpg)"); plt.axis("off")
plt.show()


In [ ]:
# Read images
target = cv2.imread(str(DATA_DIR / "MRI.jpg"))   # fixed
source = cv2.imread(str(DATA_DIR / "MRI2.jpg"))  # moving
if target is None or source is None:
    raise FileNotFoundError("MRI.jpg or MRI2.jpg not found")
# Copies for drawing points
target_vis = target.copy()
source_vis = source.copy()
# Store clicked points
pts_target = []  # on MRI.jpg
pts_source = []  # on MRI2.jpg


In [ ]:
# Show with axes
plt.figure(figsize=(6,6))
plt.imshow(target, cmap="gray")
plt.title("TARGET - read (x,y)")
plt.axis("on")
plt.show()
plt.figure(figsize=(6,6))
plt.imshow(source, cmap="gray")
plt.title("SOURCE - read (x,y)")
plt.axis("on")
plt.show()


In [ ]:
# Mouse- SOURCE window
def click_source(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        pts_source.append([x, y])
        cv2.circle(source_vis, (x, y), 5, (0,0,255), -1)
        cv2.imshow("SOURCE (MRI2)", source_vis)
# Mouse- TARGET window
def click_target(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        pts_target.append([x, y])
        cv2.circle(target_vis, (x, y), 5, (0,0,255), -1)
        cv2.imshow("TARGET (MRI)", target_vis)


In [ ]:
# Create windows
cv2.namedWindow("SOURCE (MRI2)", cv2.WINDOW_NORMAL)
cv2.namedWindow("TARGET (MRI)", cv2.WINDOW_NORMAL)
# Attach callbacks
cv2.setMouseCallback("SOURCE (MRI2)", click_source)
cv2.setMouseCallback("TARGET (MRI)", click_target)
# Show windows
cv2.imshow("SOURCE (MRI2)", source_vis)
cv2.imshow("TARGET (MRI)", target_vis)
cv2.waitKey(0)
cv2.destroyAllWindows()
print("pts_source =", pts_source)
print("pts_target =", pts_target)

In [ ]:
# Three point pairs define a 2D affine transformation
if len(pts_source) < 3 or len(pts_target) < 3:
    raise ValueError("Pick at least 3 corresponding points on each image")
# Use the first three point pairs
P_s = np.float32(pts_source[:3])  # source points
P_t = np.float32(pts_target[:3])  # target points
# A 
A = cv2.getAffineTransform(P_s, P_t)
print("Affine A:\n", A)
# Warp source into target size
h, w = target.shape[:2]
warped = cv2.warpAffine(source, A, (w, h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT,borderValue=0)
# Display the warped image
warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5,5))
plt.imshow(warped_rgb)
plt.title("Warped (MRI2 -> MRI)")
plt.axis("off")
plt.show()


In [ ]:
# Convert for plotting
target_rgb = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

# Overlay-alpha blend
alpha = 0.5
overlay = cv2.addWeighted(target, 1-alpha, warped, alpha, 0)
overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
# Display
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(target_rgb)
plt.title("Target (MRI)")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(warped_rgb)
plt.title("Warped Source (MRI2)")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay_rgb)
plt.title("Overlay (Target + Warped)")
plt.axis("off")
plt.show()


### Registration analysis

Three corresponding point pairs are sufficient to estimate a 2D affine transformation. Adding a fourth landmark can help assess alignment quality, but `getAffineTransform` estimates its matrix from exactly three pairs. The warped image is blended with the fixed image so displacement and residual misalignment are easy to inspect.


## 2. Template matching

Locate a small template in a larger shelf image using sum of squared differences (SSD) and normalized cross-correlation (NCC).


In [ ]:
# Read images
shelf_bgr = cv2.imread(str(DATA_DIR / "shelf.jpg"))
temp_bgr  = cv2.imread(str(DATA_DIR / "template.jpg"))

# Convert to grayscale
shelf = cv2.cvtColor(shelf_bgr, cv2.COLOR_BGR2GRAY)
temp  = cv2.cvtColor(temp_bgr,  cv2.COLOR_BGR2GRAY)

# Template size
th, tw = temp.shape

# Show images
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(shelf, cmap="gray"); plt.title("Shelf"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(temp,  cmap="gray"); plt.title("Template"); plt.axis("off")
plt.show()


In [ ]:
# SSD (lower is better)
res_ssd = cv2.matchTemplate(shelf, temp, cv2.TM_SQDIFF)
# Get best match
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res_ssd)
best_ssd = min_loc  
# for SSD use min_loc
print("SSD best location:", best_ssd, "min value:", min_val)
# Show response map
plt.figure(figsize=(6,5))
plt.imshow(res_ssd, cmap="hot")
plt.title("SSD Response (TM_SQDIFF)")
plt.axis("off")
plt.show()


In [ ]:
# NCC (higher is better)
res_ncc = cv2.matchTemplate(shelf, temp, cv2.TM_CCOEFF_NORMED)
# Get best match
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res_ncc)
best_ncc = max_loc  
# for NCC use max_loc
print("NCC best location:", best_ncc, "max value:", max_val)
# Show response map
plt.figure(figsize=(6,5))
plt.imshow(res_ncc, cmap="hot")
plt.title("NCC Response (TM_CCOEFF_NORMED)")
plt.axis("off")
plt.show()


In [ ]:
# Convert shelf to RGB for drawing with matplotlib
shelf_rgb = cv2.cvtColor(shelf_bgr, cv2.COLOR_BGR2RGB)
# Copy images
img_ssd = shelf_rgb.copy()
img_ncc = shelf_rgb.copy()
# Draw rectangle for SSD
x1, y1 = best_ssd
cv2.rectangle(img_ssd, (x1, y1), (x1 + tw, y1 + th), (255, 0, 0), 3)  # red
# Draw rectangle for NCC
x2, y2 = best_ncc
cv2.rectangle(img_ncc, (x2, y2), (x2 + tw, y2 + th), (0, 255, 0), 3)  # green
# Show results
plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(img_ssd); plt.title("Best Match - SSD"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(img_ncc); plt.title("Best Match - NCC"); plt.axis("off")
plt.show()


### Template-matching analysis

SSD selects the location with the lowest squared intensity difference, whereas NCC selects the location with the highest normalized correlation. NCC is generally less sensitive to uniform brightness and contrast changes. Both response maps identify the most likely template position, which is shown with a bounding box.


## 3. Iterative intensity-based registration

Define a gradient-descent procedure that updates a six-parameter affine transformation by minimizing intensity differences between fixed and warped images.


### Optimization model

At each iteration, the moving image is warped with the current affine matrix. The residual image and spatial gradients form a Jacobian for the six affine parameters. The sum-of-squared-error gradient then updates the transformation. The learning rate controls stability: a value that is too large may diverge, while a very small value converges slowly.


In [ ]:
def iterative(img_fixed, img_moving, T, iters=100, lr=1e-6):
    # img_fixed, img_moving: grayscale float32
    h, w = img_fixed.shape
    for i in range(iters):
        #warp moving- fixed
        img_warp = cv2.warpAffine(img_moving, T, (w, h), flags=cv2.INTER_LINEAR)
        #error image
        error = img_fixed - img_warp
        #gradient of warped image
        Ix = cv2.Sobel(img_warp, cv2.CV_32F, 1, 0, ksize=3)
        Iy = cv2.Sobel(img_warp, cv2.CV_32F, 0, 1, ksize=3)
        # build Jacobian 
        yy, xx = np.mgrid[0:h, 0:w]
        xx = xx.reshape(-1).astype(np.float32)
        yy = yy.reshape(-1).astype(np.float32)
        Ix = Ix.reshape(-1)
        Iy = Iy.reshape(-1)
        e  = error.reshape(-1)
        # J = [dI/da, dI/db, dI/dtx, dI/dc, dI/dd, dI/dty]
        J = np.vstack([Ix * xx,Ix * yy,Ix,Iy * xx,Iy * yy,Iy]).T
        #gradient of SSE and update
        grad = J.T @ e  # (6,)
        Tvec = T.reshape(-1) - lr * grad
        T = Tvec.reshape(2, 3)

    return T


## 4. Manual Otsu thresholding

Compute the between-class variance for every grayscale threshold, select the maximum, and create a binary coin image.


### Otsu method

Otsu thresholding separates foreground and background by maximizing between-class variance. For every candidate threshold, the grayscale histogram is divided into two classes and their probabilities and means are evaluated. The optimal threshold is the point at which the two classes are most distinct.


In [ ]:
# Part A
# Image path
IMG_PATH = DATA_DIR / "coin.png"   
# Read image
img_bgr = cv2.imread(str(IMG_PATH))
# Convert to grayscale
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
# display image
plt.imshow(gray, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")
plt.show()


In [ ]:
# Part B
def otsu_manual(gray):
    # Compute histogram
    hist = np.bincount(gray.ravel(), minlength=256)
    # Total number of pixels
    N = gray.size
    # Normalize histogram
    p = hist / N
    # Cumulative sum of probabilities
    w0 = np.cumsum(p)
    # Cumulative mean
    mu = np.cumsum(p * np.arange(256))
    # Global mean
    mu_t = mu[-1]
    # Between-class variance array
    sigma_b2 = np.zeros(256)
    # Denominator term
    denom = w0 * (1 - w0)
    # Valid positions (avoid division by zero)
    valid = denom > 1e-12
    # Compute between-class variance
    sigma_b2[valid] = ((mu_t*w0[valid] - mu[valid])**2) / denom[valid]
    # Best threshold
    T = np.argmax(sigma_b2)
    return T, sigma_b2
# Run Otsu
T_star, sigma_b2 = otsu_manual(gray)
# Print threshold
print("Optimal threshold:", T_star)


In [ ]:
# Gray levels
k = np.arange(256)
plt.figure(figsize=(9,5))
plt.plot(k, sigma_b2, linewidth=2, label=r'$\sigma_B^2(k)$')
plt.axvline(T_star,color='pink', linestyle='--', linewidth=2, label=f'Optimal k* = {T_star}')
#optimal threshold
plt.grid(True, linestyle='--', alpha=0.6)
plt.title("Between-Class Variance vs Threshold (Otsu's Method)")
plt.xlabel("Threshold k")
plt.ylabel(r'$\sigma_B^2(k)$')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Part C
# Apply threshold
binary = (gray > T_star).astype(np.uint8) * 255
# Show result
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(gray, cmap="gray")
plt.title("Original")
plt.axis("off")
plt.subplot(1,2,2)
plt.imshow(binary, cmap="gray")
plt.title("Thresholded")
plt.axis("off")
plt.show()


## 5. Hough line detection

Extract edges with Canny, construct a Hough accumulator, and draw the four strongest detected lines.


In [ ]:
# Load image
img_bgr = cv2.imread(str(DATA_DIR / "airport.tif"))
# Convert to RGB
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
# Convert to gray
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
# Show image
plt.imshow(gray, cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.show()


In [ ]:
# Canny edge detection
edges = cv2.Canny(gray, 80, 150)
# Show edges
plt.imshow(edges, cmap="gray")
plt.title("Edge Image")
plt.axis("off")
plt.show()


In [ ]:
# Part B
# Apply Hough Line Transform
lines = cv2.HoughLines(edges, 1, np.pi/180, 150)
# Create accumulator space image
h, w = edges.shape
rho_max = int(np.sqrt(h*h + w*w))
accumulator = np.zeros((2*rho_max, 180))
# Fill accumulator manually
ys, xs = np.where(edges > 0)
for i in range(len(xs)):
    x = xs[i]
    y = ys[i]
    for theta in range(180):
        t = np.deg2rad(theta)
        rho = int(x*np.cos(t) + y*np.sin(t))
        accumulator[rho + rho_max, theta] += 1
# Show Hough space
plt.figure(figsize=(8,6))
plt.imshow(accumulator, cmap="hot", aspect="auto")
plt.title("Hough Space (Accumulator)")
plt.xlabel("Theta (degrees)")
plt.ylabel("Rho")
plt.colorbar()
plt.show()


In [ ]:
# Draw the four strongest Hough lines
# If your image is grayscale, convert to BGR for colored lines
if len(img.shape) == 2:
    img_lines = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
else:
    img_lines = img.copy()

if lines is None or len(lines) < 4:
    raise ValueError("Not enough lines detected. Try lowering the Hough threshold.")

top4 = lines[:4]

# Draw top 4 lines
for i in range(4):
    rho, theta = top4[i][0]
    a = np.cos(theta)
    b = np.sin(theta)

    x0 = a * rho
    y0 = b * rho

    # Make the line long enough to cross the image
    x1 = int(x0 + 2000 * (-b))
    y1 = int(y0 + 2000 * (a))
    x2 = int(x0 - 2000 * (-b))
    y2 = int(y0 - 2000 * (a))

    # OpenCV uses BGR, so red is (0,0,255)
    cv2.line(img_lines, (x1, y1), (x2, y2), (0, 0, 255), 2)

# Show result correctly in matplotlib (convert BGR -> RGB)
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(img_lines, cv2.COLOR_BGR2RGB))
plt.title("Top 4 Detected Lines")
plt.axis("off")
plt.show()


### Hough-transform analysis

Canny detection creates a binary edge map. Each edge pixel votes for all lines that could pass through it in polar coordinates `(rho, theta)`. Peaks in the accumulator represent prominent image lines. The four strongest detections are converted back to image coordinates and drawn over the source image.


## 6. Seeded region growing

Segment cell regions from a grayscale image using interactively selected seeds and an intensity-similarity criterion. Compare results before and after Gaussian smoothing.


In [ ]:
# Load original image
img_raw = cv2.imread(str(DATA_DIR / "cells.png"), cv2.IMREAD_GRAYSCALE)
# Show original image
plt.imshow(img_raw, cmap='gray')
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
# Apply Gaussian smoothing
img_smooth = cv2.GaussianBlur(img_raw, (5,5), 0)
plt.imshow(img_smooth, cmap='gray')
plt.title("Smoothed Image")
plt.axis("off")
plt.show()

In [ ]:
# Prepare for seed selection
seeds = []
display = cv2.cvtColor(img_smooth, cv2.COLOR_GRAY2BGR)
# select seeds
def mouse_click(event, x, y, flags, param):
    global seeds, display
    if event == cv2.EVENT_LBUTTONDOWN:  
        seeds.append((x, y))            
        # Draw red point
        cv2.circle(display, (x, y), 3, (0, 0, 255), -1)
        cv2.imshow("Select Seeds", display)
# Region Growing (multi-seed)
def region_growing_multi(img, seeds, threshold=15):
    h, w = img.shape
    segmented = np.zeros_like(img, dtype=np.uint8)
    visited   = np.zeros_like(img, dtype=bool)
    stack = []
    region_sum = 0
    # Initialize with seeds
    for (x, y) in seeds:
        stack.append((y, x))
        segmented[y, x] = 255
        visited[y, x] = True
        region_sum += int(img[y, x])
    # Initial mean value
    region_mean = region_sum / max(len(seeds), 1)
    # Grow region
    while stack:
        y, x = stack.pop()
        # Check 8 neighbors
        for ny in range(y-1, y+2):
            for nx in range(x-1, x+2):
                if 0 <= ny < h and 0 <= nx < w and not visited[ny, nx]:
                    visited[ny, nx] = True
                    # Similarity check
                    if abs(int(img[ny, nx]) - region_mean) <= threshold:
                        segmented[ny, nx] = 255
                        stack.append((ny, nx))
    return segmented

In [ ]:
# Select seeds
cv2.imshow("Select Seeds", display)
cv2.setMouseCallback("Select Seeds", mouse_click)
print("Click to select seeds. Press ENTER or 'q' when done.")
while True:
    key = cv2.waitKey(20) & 0xFF
    if key == 13 or key == ord('q'):   # Enter or q
        break
cv2.destroyAllWindows()
print("Selected seeds:", seeds)
if len(seeds) == 0:
    raise ValueError("No seeds selected")
# Run Region Growing
seg_raw    = region_growing_multi(img_raw, seeds, threshold=15)
seg_smooth = region_growing_multi(img_smooth, seeds, threshold=15)
# Show comparison
plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
plt.imshow(img_raw, cmap='gray')
plt.title("Original Image")
plt.axis("off")
plt.subplot(1,3,2)
plt.imshow(seg_raw, cmap='gray')
plt.title("Region Growing (No Smoothing)")
plt.axis("off")
plt.subplot(1,3,3)
plt.imshow(seg_smooth, cmap='gray')
plt.title("Region Growing (With Smoothing)")
plt.axis("off")
plt.show()


### Region-growing analysis

Region growing starts from user-selected seed pixels and repeatedly adds connected neighbors whose intensity is sufficiently close to the region mean. Gaussian smoothing reduces local noise and usually produces a more continuous segmentation. The result depends on both seed placement and the similarity threshold.
